# BioBERT Fine-tuning for Medical Abstract Classification (Leakage-Free)
Run this notebook in Google Colab with GPU runtime.
This version incorporates explicit keyword stripping to prevent label leakage.


In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy matplotlib seaborn tqdm huggingface-hub -q


In [ ]:
import torch
import random
import numpy as np
import re

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('/content/drive/MyDrive/medical_classification/results/', exist_ok=True)
os.makedirs('/content/drive/MyDrive/medical_classification/model/', exist_ok=True)
print('Directories created.')


In [ ]:
CONFIG = {
    'MODEL_NAME': 'dmis-lab/biobert-base-cased-v1.2',
    'MAX_LENGTH': 256,
    'BATCH_SIZE': 16,
    'EPOCHS': 3,
    'LEARNING_RATE': 2e-5,
    'WEIGHT_DECAY': 0.01,
    'SEED': 42,
    'NUM_LABELS': 3,
    'LABEL_MAP': {'Diagnosis': 0, 'Prevention': 1, 'Treatment': 2},
    'OUTPUT_DIR': '/content/drive/MyDrive/medical_classification/model/biobert_checkpoints/',
    'RESULTS_DIR': '/content/drive/MyDrive/medical_classification/results/'
}


In [ ]:
import pandas as pd
import os
import re

print('Loading pubmed_rct dataset from Google Drive...')
data_dir = '/content/drive/MyDrive/medical_classification/data/'
dfs = []
for split in ['train.csv', 'dev.csv', 'test.csv']:
    path = os.path.join(data_dir, split)
    if os.path.exists(path):
        dfs.append(pd.read_csv(path))
df_all = pd.concat(dfs, ignore_index=True)

print('Reconstructing abstracts...')
df_all = df_all.sort_values(by=['abstract_id', 'line_number'])
df_abs = df_all.groupby('abstract_id')['abstract_text'].apply(lambda x: ' '.join(x)).reset_index()

treatment_kw = ['intervention', 'trial', 'therapy', 'drug', 'dose', 'randomised', 'placebo', 'administered', 'efficacy']
diagnosis_kw = ['screening', 'diagnostic', 'sensitivity', 'specificity', 'test', 'accuracy', 'imaging', 'biopsy', 'detection']
prevention_kw = ['vaccine', 'prevention', 'prophylaxis', 'risk reduction', 'protective', 'immunisation', 'incidence']
all_kws = treatment_kw + diagnosis_kw + prevention_kw + ['diagnosis', 'treatment', 'prevention']
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_kws)) + r')\b', flags=re.IGNORECASE)

labels = []
cleaned_texts = []
for text in df_abs['abstract_text']:
    text_lower = text.lower()
    counts = {
        'Treatment': sum(text_lower.count(kw) for kw in treatment_kw),
        'Diagnosis': sum(text_lower.count(kw) for kw in diagnosis_kw),
        'Prevention': sum(text_lower.count(kw) for kw in prevention_kw)
    }
    max_count = max(counts.values())
    if max_count == 0:
        labels.append('Treatment')
    else:
        top = [k for k, v in counts.items() if v == max_count]
        labels.append('Treatment' if len(top) > 1 else top[0])
    
    # Data Leakage Fix: Strip keywords
    clean_text = pattern.sub('', text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    cleaned_texts.append(clean_text)
        
df_abs['label'] = labels
df_abs['abstract_text'] = cleaned_texts
df_abs['label_id'] = df_abs['label'].map(CONFIG['LABEL_MAP'])

print('Stratifying dataset...')
target_size = 20000
classes = df_abs['label'].unique()
samples_per_class = target_size // len(classes)
remainder = target_size % len(classes)

balanced_dfs = []
for i, c in enumerate(classes):
    n_samples = samples_per_class + (1 if i < remainder else 0)
    class_df = df_abs[df_abs['label'] == c]
    sampled = class_df.sample(n=n_samples, replace=(n_samples > len(class_df)), random_state=42)
    balanced_dfs.append(sampled)
    
df_balanced = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
print('Class distribution after balancing:')
print(df_balanced['label'].value_counts())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix: BioBERT')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(CONFIG['RESULTS_DIR'] + 'biobert_confusion_matrix.png', dpi=300)
plt.show()


In [ ]:
history = trainer.state.log_history
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]
epochs_eval = range(1, len(eval_loss) + 1)

plt.figure(figsize=(8, 6))
plt.plot(epochs_eval, eval_loss, label='Validation Loss', marker='o')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig(CONFIG['RESULTS_DIR'] + 'biobert_loss_curves.png', dpi=300)
plt.show()


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_true_bin = label_binarize(y_true, classes=[0,1,2])
n_classes = 3

plt.figure(figsize=(8, 6))
colors = ['aqua', 'darkorange', 'cornflowerblue']
for i, color in zip(range(n_classes), colors):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'ROC curve of {target_names[i]} (area = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('One-vs-Rest ROC Curve: BioBERT')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(CONFIG['RESULTS_DIR'] + 'biobert_roc_curve.png', dpi=300)
plt.show()
